# Dimensionality Reduction of Engineering Dataset using PCA
This notebook analyzes the UCI Concrete Compressive Strength dataset. Upload `Concrete_Data.xls` when prompted.

In [ ]:
%pip -q install xlrd seaborn scikit-learn
from google.colab import files
uploaded = files.upload()
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from pathlib import Path
file_path = next(iter(uploaded))
df = pd.read_excel(file_path)
print('Dataset shape:', df.shape)
display(df.head())
print('Missing values:', int(df.isna().sum().sum()))


## PCA, visualization, and regression comparison
The dataset's final column is treated as the target; the preceding eight columns are input features.

In [ ]:
X = df.iloc[:, :-1].copy()
y = df.iloc[:, -1].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
pca = PCA().fit(X_train_s)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)
print('Explained variance by component (%):', np.round(evr*100, 2))
print('Cumulative variance (%):', np.round(cum*100, 2))
print('Components for >=95% variance:', int(np.argmax(cum >= 0.95) + 1))

plt.figure(figsize=(8,4))
plt.plot(range(1, len(evr)+1), evr*100, marker='o', label='Individual')
plt.plot(range(1, len(cum)+1), cum*100, marker='s', label='Cumulative')
plt.xlabel('Principal component'); plt.ylabel('Explained variance (%)'); plt.title('PCA Scree Plot'); plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(8,6))
proj = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X))
sc = plt.scatter(proj[:,0], proj[:,1], c=y, cmap='viridis', alpha=.75)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Dataset projected onto PC1 and PC2'); plt.colorbar(sc, label=y.name); plt.show()

results = []
base = LinearRegression().fit(X_train, y_train)
base_pred = base.predict(X_test)
results.append(('Original 8 features', np.nan, r2_score(y_test, base_pred), mean_squared_error(y_test, base_pred)**0.5))
for k in range(2, X_train.shape[1]+1):
    pk = PCA(n_components=k)
    A = pk.fit_transform(X_train_s)
    B = pk.transform(X_test_s)
    model = LinearRegression().fit(A, y_train)
    pred = model.predict(B)
    results.append((f'PCA k={k}', float(pk.explained_variance_ratio_.sum()), r2_score(y_test, pred), mean_squared_error(y_test, pred)**0.5))
result_df = pd.DataFrame(results, columns=['Model','Variance retained','Test R2','Test RMSE'])
display(result_df)

plt.figure(figsize=(8,4))
pca_rows = result_df[result_df['Model'].str.startswith('PCA')]
plt.plot([int(s.split('=')[1]) for s in pca_rows['Model']], pca_rows['Test R2'], marker='o')
plt.xlabel('Number of components'); plt.ylabel('Test R²'); plt.title('Regression performance vs PCA components'); plt.grid(True); plt.show()

plt.figure(figsize=(9,6)); sns.heatmap(X.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f'); plt.title('Feature correlation heatmap'); plt.show()


## Notes
Use the actual output from your run when reporting metrics. Results may vary slightly if split settings or software versions differ.